In [ ]:
# Required imports
from typing import List, Dict, Any, Optional, Callable, Union, get_type_hints
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
from enum import Enum
import json
import re
import inspect

print("✅ Imports successful")

## 1. What are Tools?

**Tools** are functions that an LLM can call to:
- Access external data (search, APIs)
- Perform calculations
- Take actions (send emails, update databases)
- Interact with the world

The LLM doesn't execute tools directly - it outputs a structured request,
and our system executes the tool and returns results.

In [ ]:
# The LLM sees a description like this:
TOOL_EXAMPLE = """
Tool: calculator
Description: Evaluate mathematical expressions
Parameters:
  - expression (string, required): The math expression to evaluate
  
Example usage:
  Action: calculator
  Action Input: {"expression": "25 * 4 + 10"}
  Observation: 110
"""

print("🔧 How the LLM sees a tool:")
print(TOOL_EXAMPLE)

## 2. Tool Parameter Schema

Tools need well-defined parameters for validation:

In [ ]:
class ParameterType(Enum):
    """Supported parameter types."""
    STRING = "string"
    INTEGER = "integer"
    FLOAT = "float"
    BOOLEAN = "boolean"
    ARRAY = "array"
    OBJECT = "object"


@dataclass
class ToolParameter:
    """Definition of a single tool parameter."""
    name: str
    param_type: ParameterType
    description: str
    required: bool = True
    default: Any = None
    enum_values: Optional[List[Any]] = None  # Allowed values
    
    def validate(self, value: Any) -> tuple[bool, str]:
        """Validate a value against this parameter definition."""
        if value is None:
            if self.required and self.default is None:
                return False, f"Parameter '{self.name}' is required"
            return True, ""
        
        # Type checking
        type_checks = {
            ParameterType.STRING: lambda v: isinstance(v, str),
            ParameterType.INTEGER: lambda v: isinstance(v, int) and not isinstance(v, bool),
            ParameterType.FLOAT: lambda v: isinstance(v, (int, float)),
            ParameterType.BOOLEAN: lambda v: isinstance(v, bool),
            ParameterType.ARRAY: lambda v: isinstance(v, list),
            ParameterType.OBJECT: lambda v: isinstance(v, dict),
        }
        
        if not type_checks[self.param_type](value):
            return False, f"Parameter '{self.name}' must be type {self.param_type.value}"
        
        # Enum validation
        if self.enum_values and value not in self.enum_values:
            return False, f"Parameter '{self.name}' must be one of {self.enum_values}"
        
        return True, ""
    
    def to_json_schema(self) -> Dict[str, Any]:
        """Convert to JSON schema format (OpenAI compatible)."""
        schema = {
            "type": self.param_type.value,
            "description": self.description
        }
        if self.enum_values:
            schema["enum"] = self.enum_values
        if self.default is not None:
            schema["default"] = self.default
        return schema

print("✅ ToolParameter class defined")

In [ ]:
# Example parameter definitions

expression_param = ToolParameter(
    name="expression",
    param_type=ParameterType.STRING,
    description="A mathematical expression to evaluate (e.g., '2 + 2 * 3')",
    required=True
)

# Test validation
print("Validating 'expression' parameter:")
print(f"  '25 * 4': {expression_param.validate('25 * 4')}")
print(f"  123 (int): {expression_param.validate(123)}")
print(f"  None: {expression_param.validate(None)}")

# Enum parameter example
unit_param = ToolParameter(
    name="unit",
    param_type=ParameterType.STRING,
    description="Temperature unit",
    required=False,
    default="celsius",
    enum_values=["celsius", "fahrenheit", "kelvin"]
)

print("\nValidating 'unit' parameter:")
print(f"  'celsius': {unit_param.validate('celsius')}")
print(f"  'rankine': {unit_param.validate('rankine')}")

## 3. Tool Base Class

Now let's define the full Tool structure:

In [ ]:
@dataclass
class ToolResult:
    """Result from executing a tool."""
    success: bool
    output: Any
    error: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def __str__(self) -> str:
        if self.success:
            return str(self.output)
        return f"Error: {self.error}"


class Tool(ABC):
    """
    Abstract base class for all tools.
    
    Tools are callable units that extend the agent's capabilities.
    """
    
    def __init__(
        self,
        name: str,
        description: str,
        parameters: List[ToolParameter] = None
    ):
        self.name = name
        self.description = description
        self.parameters = parameters or []
        self._param_lookup = {p.name: p for p in self.parameters}
    
    @abstractmethod
    def _execute(self, **kwargs) -> Any:
        """Execute the tool. Override in subclasses."""
        pass
    
    def run(self, **kwargs) -> ToolResult:
        """Validate and execute the tool."""
        # Apply defaults
        for param in self.parameters:
            if param.name not in kwargs and param.default is not None:
                kwargs[param.name] = param.default
        
        # Validate parameters
        for param in self.parameters:
            value = kwargs.get(param.name)
            valid, error = param.validate(value)
            if not valid:
                return ToolResult(success=False, output=None, error=error)
        
        # Execute
        try:
            result = self._execute(**kwargs)
            return ToolResult(success=True, output=result)
        except Exception as e:
            return ToolResult(success=False, output=None, error=str(e))
    
    def get_schema(self) -> Dict[str, Any]:
        """Get OpenAI-compatible function schema."""
        properties = {}
        required = []
        
        for param in self.parameters:
            properties[param.name] = param.to_json_schema()
            if param.required:
                required.append(param.name)
        
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": properties,
                    "required": required
                }
            }
        }
    
    def get_description(self) -> str:
        """Get human-readable description for prompts."""
        lines = [f"Tool: {self.name}", f"Description: {self.description}", "Parameters:"]
        for param in self.parameters:
            req = "required" if param.required else "optional"
            lines.append(f"  - {param.name} ({param.param_type.value}, {req}): {param.description}")
        return "\n".join(lines)
    
    def __call__(self, **kwargs) -> ToolResult:
        """Allow calling the tool directly."""
        return self.run(**kwargs)

print("✅ Tool base class defined")

## 4. Built-in Tools

Let's implement some common tools:

In [ ]:
class CalculatorTool(Tool):
    """
    A safe calculator tool for mathematical expressions.
    """
    
    # Allowed operations for security
    ALLOWED_CHARS = set('0123456789+-*/().% ')
    ALLOWED_FUNCS = {'abs', 'round', 'min', 'max', 'pow', 'sqrt'}
    
    def __init__(self):
        super().__init__(
            name="calculator",
            description="Evaluate mathematical expressions. Supports +, -, *, /, %, (), and functions like abs, round, min, max, pow, sqrt.",
            parameters=[
                ToolParameter(
                    name="expression",
                    param_type=ParameterType.STRING,
                    description="The mathematical expression to evaluate (e.g., '25 * 4 + 10')",
                    required=True
                )
            ]
        )
    
    def _execute(self, expression: str) -> float:
        """Safely evaluate the expression."""
        # Clean the expression
        expr = expression.replace('^', '**')  # Support ^ for power
        
        # Security check - only allow safe characters
        for char in expr:
            if char not in self.ALLOWED_CHARS and not char.isalpha():
                raise ValueError(f"Invalid character: {char}")
        
        # Check for function calls
        import math
        safe_dict = {
            'abs': abs,
            'round': round,
            'min': min,
            'max': max,
            'pow': pow,
            'sqrt': math.sqrt,
        }
        
        try:
            result = eval(expr, {"__builtins__": {}}, safe_dict)
            return float(result)
        except Exception as e:
            raise ValueError(f"Cannot evaluate expression: {e}")

print("✅ CalculatorTool defined")

In [ ]:
# Test the calculator
calc = CalculatorTool()

print("🔢 Calculator Tests:")
print("-" * 40)

expressions = [
    "25 * 4 + 10",
    "(100 - 20) / 4",
    "2 ** 10",
    "sqrt(16)",
    "max(10, 20, 15)"
]

for expr in expressions:
    result = calc.run(expression=expr)
    print(f"  {expr} = {result}")

In [ ]:
class SearchTool(Tool):
    """
    A mock search tool (simulates web search).
    In production, this would call a real search API.
    """
    
    # Mock database of facts
    MOCK_DATA = {
        "python": "Python is a high-level programming language created by Guido van Rossum in 1991.",
        "paris": "Paris is the capital of France with a population of about 2.1 million.",
        "einstein": "Albert Einstein (1879-1955) was a theoretical physicist who developed the theory of relativity.",
        "moon": "The Moon is Earth's only natural satellite, about 384,400 km away.",
        "coffee": "Coffee is a brewed drink made from roasted coffee beans, native to tropical Africa.",
    }
    
    def __init__(self):
        super().__init__(
            name="search",
            description="Search for information on a topic. Returns relevant facts and information.",
            parameters=[
                ToolParameter(
                    name="query",
                    param_type=ParameterType.STRING,
                    description="The search query",
                    required=True
                ),
                ToolParameter(
                    name="max_results",
                    param_type=ParameterType.INTEGER,
                    description="Maximum number of results to return",
                    required=False,
                    default=3
                )
            ]
        )
    
    def _execute(self, query: str, max_results: int = 3) -> str:
        """Search for information."""
        query_lower = query.lower()
        results = []
        
        for key, value in self.MOCK_DATA.items():
            if key in query_lower or any(word in key for word in query_lower.split()):
                results.append(value)
        
        if not results:
            return f"No results found for: {query}"
        
        return "\n".join(results[:max_results])

print("✅ SearchTool defined")

In [ ]:
# Test the search tool
search = SearchTool()

print("🔍 Search Tests:")
print("-" * 40)

queries = ["Tell me about Python", "Paris facts", "who is Einstein"]

for q in queries:
    result = search.run(query=q)
    print(f"\nQuery: {q}")
    print(f"Result: {result}")

In [ ]:
class DateTimeTool(Tool):
    """Tool for getting current date/time and date calculations."""
    
    def __init__(self):
        super().__init__(
            name="datetime",
            description="Get current date/time or perform date calculations",
            parameters=[
                ToolParameter(
                    name="action",
                    param_type=ParameterType.STRING,
                    description="The action to perform",
                    required=True,
                    enum_values=["now", "date", "time", "weekday", "add_days"]
                ),
                ToolParameter(
                    name="days",
                    param_type=ParameterType.INTEGER,
                    description="Number of days to add (for add_days action)",
                    required=False,
                    default=0
                )
            ]
        )
    
    def _execute(self, action: str, days: int = 0) -> str:
        from datetime import datetime, timedelta
        
        now = datetime.now()
        
        if action == "now":
            return now.strftime("%Y-%m-%d %H:%M:%S")
        elif action == "date":
            return now.strftime("%Y-%m-%d")
        elif action == "time":
            return now.strftime("%H:%M:%S")
        elif action == "weekday":
            return now.strftime("%A")
        elif action == "add_days":
            future = now + timedelta(days=days)
            return future.strftime("%Y-%m-%d")
        else:
            raise ValueError(f"Unknown action: {action}")

# Test it
dt_tool = DateTimeTool()
print("📅 DateTime Tests:")
print(f"  Now: {dt_tool.run(action='now')}")
print(f"  Date: {dt_tool.run(action='date')}")
print(f"  Weekday: {dt_tool.run(action='weekday')}")
print(f"  +7 days: {dt_tool.run(action='add_days', days=7)}")

## 5. Tool Registry

A **registry** manages multiple tools and allows lookup:

In [ ]:
class ToolRegistry:
    """
    Registry for managing and discovering tools.
    """
    
    def __init__(self):
        self._tools: Dict[str, Tool] = {}
        self._categories: Dict[str, List[str]] = {}  # category -> tool names
    
    def register(self, tool: Tool, category: str = "general") -> None:
        """Register a tool."""
        if tool.name in self._tools:
            raise ValueError(f"Tool '{tool.name}' already registered")
        
        self._tools[tool.name] = tool
        
        if category not in self._categories:
            self._categories[category] = []
        self._categories[category].append(tool.name)
        
        print(f"  ✅ Registered: {tool.name} ({category})")
    
    def get(self, name: str) -> Optional[Tool]:
        """Get a tool by name."""
        return self._tools.get(name)
    
    def execute(self, name: str, **kwargs) -> ToolResult:
        """Execute a tool by name."""
        tool = self.get(name)
        if not tool:
            return ToolResult(
                success=False,
                output=None,
                error=f"Tool '{name}' not found. Available: {list(self._tools.keys())}"
            )
        return tool.run(**kwargs)
    
    def list_tools(self, category: Optional[str] = None) -> List[str]:
        """List all tools, optionally filtered by category."""
        if category:
            return self._categories.get(category, [])
        return list(self._tools.keys())
    
    def list_categories(self) -> List[str]:
        """List all categories."""
        return list(self._categories.keys())
    
    def get_all_schemas(self) -> List[Dict[str, Any]]:
        """Get OpenAI-compatible schemas for all tools."""
        return [tool.get_schema() for tool in self._tools.values()]
    
    def get_tools_description(self) -> str:
        """Get human-readable description of all tools."""
        descriptions = []
        for tool in self._tools.values():
            descriptions.append(tool.get_description())
        return "\n\n".join(descriptions)
    
    def __len__(self) -> int:
        return len(self._tools)
    
    def __contains__(self, name: str) -> bool:
        return name in self._tools

print("✅ ToolRegistry defined")

In [ ]:
# Create a registry and register tools

registry = ToolRegistry()

print("Registering tools:")
registry.register(CalculatorTool(), category="math")
registry.register(SearchTool(), category="information")
registry.register(DateTimeTool(), category="utility")

print(f"\nTotal tools: {len(registry)}")
print(f"Categories: {registry.list_categories()}")

In [ ]:
# Using the registry

print("📝 Tool Descriptions for LLM:")
print("=" * 60)
print(registry.get_tools_description())

In [ ]:
# Execute tools through registry

print("\n🔧 Executing tools via registry:")
print("-" * 40)

# Math
result = registry.execute("calculator", expression="100 * 1.15")
print(f"Calculator: 100 * 1.15 = {result}")

# Search
result = registry.execute("search", query="moon distance")
print(f"Search: {result}")

# DateTime
result = registry.execute("datetime", action="weekday")
print(f"Today is: {result}")

# Unknown tool
result = registry.execute("unknown_tool")
print(f"Unknown: {result}")

## 6. Function-based Tool Creation

Sometimes it's easier to create tools from simple functions:

In [ ]:
def tool_from_function(func: Callable, description: str = None) -> Tool:
    """
    Create a Tool from a regular Python function.
    Uses type hints and docstrings for parameter info.
    """
    
    class FunctionTool(Tool):
        def _execute(self, **kwargs):
            return func(**kwargs)
    
    # Get function info
    sig = inspect.signature(func)
    hints = get_type_hints(func) if hasattr(func, '__annotations__') else {}
    
    # Build parameters
    parameters = []
    for param_name, param in sig.parameters.items():
        # Determine type
        hint = hints.get(param_name, str)
        if hint == int:
            ptype = ParameterType.INTEGER
        elif hint == float:
            ptype = ParameterType.FLOAT
        elif hint == bool:
            ptype = ParameterType.BOOLEAN
        elif hint == list:
            ptype = ParameterType.ARRAY
        elif hint == dict:
            ptype = ParameterType.OBJECT
        else:
            ptype = ParameterType.STRING
        
        # Check if required
        required = param.default == inspect.Parameter.empty
        default = None if required else param.default
        
        parameters.append(ToolParameter(
            name=param_name,
            param_type=ptype,
            description=f"Parameter: {param_name}",
            required=required,
            default=default
        ))
    
    return FunctionTool(
        name=func.__name__,
        description=description or func.__doc__ or f"Function: {func.__name__}",
        parameters=parameters
    )

print("✅ tool_from_function defined")

In [ ]:
# Create tools from simple functions

def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert Celsius temperature to Fahrenheit."""
    return celsius * 9/5 + 32

def word_count(text: str) -> int:
    """Count the number of words in text."""
    return len(text.split())

def greeting(name: str, formal: bool = False) -> str:
    """Generate a greeting message."""
    if formal:
        return f"Good day, {name}. How may I assist you?"
    return f"Hey {name}! What's up?"

# Convert to tools
temp_tool = tool_from_function(celsius_to_fahrenheit)
word_tool = tool_from_function(word_count)
greet_tool = tool_from_function(greeting)

# Register them
print("Registering function-based tools:")
registry.register(temp_tool, category="utility")
registry.register(word_tool, category="text")
registry.register(greet_tool, category="utility")

In [ ]:
# Test function-based tools

print("\n🔧 Testing function-based tools:")
print("-" * 40)

# Temperature
result = registry.execute("celsius_to_fahrenheit", celsius=25)
print(f"25°C = {result}°F")

# Word count
result = registry.execute("word_count", text="This is a test sentence with seven words")
print(f"Word count: {result}")

# Greeting
result = registry.execute("greeting", name="Alice")
print(f"Casual: {result}")

result = registry.execute("greeting", name="Dr. Smith", formal=True)
print(f"Formal: {result}")

## 7. Tool Call Parsing

When the LLM outputs tool calls, we need to parse them:

In [ ]:
@dataclass
class ToolCall:
    """A parsed tool call from LLM output."""
    tool_name: str
    arguments: Dict[str, Any]
    raw_text: str = ""


class ToolCallParser:
    """
    Parse tool calls from LLM output text.
    Supports multiple formats.
    """
    
    @staticmethod
    def parse_react_format(text: str) -> List[ToolCall]:
        """
        Parse ReAct format:
        Action: tool_name
        Action Input: {"arg": "value"}
        """
        calls = []
        
        # Find Action/Action Input pairs
        action_pattern = r'Action:\s*(\w+)\s*Action Input:\s*(\{.*?\}|.+?)(?=\n(?:Thought|Action|Observation|$)|$)'
        
        matches = re.findall(action_pattern, text, re.DOTALL | re.IGNORECASE)
        
        for tool_name, args_str in matches:
            try:
                # Try parsing as JSON
                args = json.loads(args_str.strip())
            except json.JSONDecodeError:
                # Fallback: treat as single string argument
                args = {"input": args_str.strip()}
            
            calls.append(ToolCall(
                tool_name=tool_name.strip(),
                arguments=args,
                raw_text=f"Action: {tool_name}\nAction Input: {args_str}"
            ))
        
        return calls
    
    @staticmethod
    def parse_function_call_format(text: str) -> List[ToolCall]:
        """
        Parse function call format:
        <function_call> tool_name(arg1=value1, arg2=value2) </function_call>
        """
        calls = []
        
        pattern = r'<function_call>\s*(\w+)\((.*)\)\s*</function_call>'
        matches = re.findall(pattern, text, re.DOTALL)
        
        for tool_name, args_str in matches:
            args = {}
            if args_str.strip():
                # Parse key=value pairs
                for pair in args_str.split(','):
                    if '=' in pair:
                        key, value = pair.split('=', 1)
                        key = key.strip()
                        value = value.strip().strip('"').strip("'")
                        # Try to convert to appropriate type
                        try:
                            args[key] = json.loads(value)
                        except:
                            args[key] = value
            
            calls.append(ToolCall(
                tool_name=tool_name,
                arguments=args,
                raw_text=f"<function_call> {tool_name}({args_str}) </function_call>"
            ))
        
        return calls

print("✅ ToolCallParser defined")

In [ ]:
# Test parsing

# ReAct format
react_text = """
Thought: I need to calculate the tip amount
Action: calculator
Action Input: {"expression": "45 * 0.15"}
Observation: 6.75
Thought: Now I know the answer
Final Answer: The 15% tip on $45 is $6.75
"""

calls = ToolCallParser.parse_react_format(react_text)
print("📝 Parsed ReAct format:")
for call in calls:
    print(f"  Tool: {call.tool_name}")
    print(f"  Args: {call.arguments}")

# Function call format
func_text = """
Let me search for that:
<function_call> search(query="Python programming", max_results=5) </function_call>
"""

calls = ToolCallParser.parse_function_call_format(func_text)
print("\n📝 Parsed Function Call format:")
for call in calls:
    print(f"  Tool: {call.tool_name}")
    print(f"  Args: {call.arguments}")

## 8. Tool Executor

Putting it all together - execute tool calls from LLM output:

In [ ]:
class ToolExecutor:
    """
    Execute tool calls from LLM output.
    """
    
    def __init__(self, registry: ToolRegistry):
        self.registry = registry
        self.parser = ToolCallParser()
    
    def execute_from_text(
        self, 
        text: str, 
        format: str = "react"
    ) -> List[Dict[str, Any]]:
        """
        Parse and execute tool calls from LLM output.
        """
        # Parse
        if format == "react":
            calls = self.parser.parse_react_format(text)
        elif format == "function":
            calls = self.parser.parse_function_call_format(text)
        else:
            raise ValueError(f"Unknown format: {format}")
        
        # Execute each call
        results = []
        for call in calls:
            result = self.registry.execute(call.tool_name, **call.arguments)
            results.append({
                "tool": call.tool_name,
                "arguments": call.arguments,
                "result": result,
                "success": result.success
            })
        
        return results
    
    def execute_call(self, call: ToolCall) -> ToolResult:
        """Execute a single tool call."""
        return self.registry.execute(call.tool_name, **call.arguments)

print("✅ ToolExecutor defined")

In [ ]:
# Test full execution pipeline

executor = ToolExecutor(registry)

# Simulate LLM output with tool calls
llm_output = """
Let me help you with that calculation.

Thought: I need to calculate the total cost with tax
Action: calculator
Action Input: {"expression": "99.99 * 1.08"}
Observation: [will be filled]

Thought: I should also check what day it is
Action: datetime
Action Input: {"action": "weekday"}
Observation: [will be filled]
"""

print("🔄 Executing tool calls from LLM output:")
print("=" * 60)

results = executor.execute_from_text(llm_output, format="react")

for r in results:
    print(f"\n📌 {r['tool']}")
    print(f"   Arguments: {r['arguments']}")
    print(f"   Result: {r['result']}")
    print(f"   Success: {'✅' if r['success'] else '❌'}")

## Summary

In this notebook, we built a complete tool system:

| Component | Purpose |
|-----------|--------|
| `ToolParameter` | Define and validate parameters |
| `Tool` | Base class for all tools |
| `ToolResult` | Standardized results |
| `ToolRegistry` | Manage multiple tools |
| `ToolCallParser` | Parse LLM tool calls |
| `ToolExecutor` | Execute parsed calls |

### Built-in Tools:
- **Calculator**: Safe math evaluation
- **Search**: Information lookup
- **DateTime**: Date/time operations

### Key Features:
- Type validation
- Default values
- Enum constraints
- Error handling
- OpenAI-compatible schemas
- Function-to-tool conversion